# YOLOv8 Torch-Pruning Starter

This notebook is a small first experiment for structured channel pruning on YOLOv8 before finetuning.

The goal here is not to get the best compressed model yet. The goal is to verify that we can load YOLOv8, build a dependency graph, prune a small percentage of channels, and run a forward pass afterward.

In [ ]:
# Install these once if your environment does not already have them.
# %pip install ultralytics torch-pruning

# Torch-Pruning performs structural pruning: it removes channels and updates
# connected layers together, instead of only masking weights to zero.

In [ ]:
import copy
from pathlib import Path

import torch
import torch_pruning as tp
from ultralytics import YOLO
from ultralytics.nn.modules import Detect


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHTS = "yolov8n.pt"      # Start small while testing the pruning workflow.
IMG_SIZE = 640
PRUNING_RATIO = 0.10        # Remove 10% of prunable channels for the first smoke test.
OUTPUT_PATH = Path("yolov8n_pruned_sample.pt")

torch.manual_seed(0)
print(f"Using device: {DEVICE}")

In [ ]:
# Load the Ultralytics wrapper, then take out the underlying PyTorch nn.Module.
# Keeping the wrapper around is useful later for validation/training APIs, but
# Torch-Pruning works directly on torch.nn.Module.
yolo = YOLO(WEIGHTS)
model = yolo.model.to(DEVICE).eval()

# This dummy input lets Torch-Pruning trace the computation graph and discover
# which Conv/BatchNorm/channel dimensions depend on each other.
example_inputs = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

# Count baseline compute and parameter size before pruning.
base_macs, base_params = tp.utils.count_ops_and_params(model, example_inputs)
print(f"Before pruning: {base_params / 1e6:.2f}M params, {base_macs / 1e9:.2f}G MACs")

In [ ]:
# YOLO detection heads produce task-specific outputs. For a simple first pass,
# keep Detect modules untouched and prune the feature extraction layers around them.
# This reduces the chance of breaking output tensor shapes before we build the
# finetuning/validation loop.
ignored_layers = []
for module in model.modules():
    if isinstance(module, Detect):
        ignored_layers.append(module)

# MagnitudeImportance ranks channels by weight magnitude. Lower-magnitude
# channels are treated as less important and are removed first.
importance = tp.importance.MagnitudeImportance(p=2)

# MetaPruner builds a dependency graph from example_inputs. When it removes a
# channel from one layer, it also updates the following dependent layers.
pruner = tp.pruner.MetaPruner(
    model,
    example_inputs,
    importance=importance,
    pruning_ratio=PRUNING_RATIO,
    ignored_layers=ignored_layers,
    global_pruning=False,  # False means each layer is pruned locally.
    round_to=8,            # Hardware-friendly channel counts, useful on GPUs.
)

# Apply one pruning step. For a first experiment, keep this conservative.
pruner.step()

In [ ]:
# Check the new model size and make sure a forward pass still works.
model.eval()
pruned_macs, pruned_params = tp.utils.count_ops_and_params(model, example_inputs)

with torch.no_grad():
    outputs = model(example_inputs)

print(f"After pruning:  {pruned_params / 1e6:.2f}M params, {pruned_macs / 1e9:.2f}G MACs")
print(f"Param change:   {(1 - pruned_params / base_params) * 100:.1f}% smaller")
print(f"MAC change:     {(1 - pruned_macs / base_macs) * 100:.1f}% fewer MACs")
print("Forward pass OK")

In [ ]:
# Save a full PyTorch checkpoint for later experiments.
# Structural pruning changes layer shapes, so saving only state_dict is often
# not enough unless you also recreate the exact pruned architecture.
checkpoint = {
    "model": copy.deepcopy(model).cpu(),
    "weights": WEIGHTS,
    "img_size": IMG_SIZE,
    "pruning_ratio": PRUNING_RATIO,
}
torch.save(checkpoint, OUTPUT_PATH)
print(f"Saved pruned checkpoint to: {OUTPUT_PATH.resolve()}")

## Next Step

For the finetuning stage, the usual flow is:

1. Validate the unpruned baseline mAP and latency.
2. Prune conservatively, for example 5-20% first.
3. Run a short finetune to recover accuracy.
4. Compare mAP, parameter count, MACs, model size, and real inference latency.

Pruning can reduce parameters/MACs without always improving real latency, so the final check should include timing on the target device.